In [3]:
import requests
import json
import re
import pandas as pd


def get_uniprot(id: str):
    endpoint = "https://rest.uniprot.org/uniprotkb/accessions"
    resp = requests.get(endpoint, params={'accessions': id})
    return resp


def uniprot_parse_response(resp):

    data = resp.json()
    results = data.get("results", [])

    output = {}

    for val in results:

        acc = val.get('primaryAccession')
        species = val.get('organism', {}).get('scientificName')
        gene = val.get('genes')
        seq = val.get('sequence')

        output[acc] = {
            'organism': species,
            'geneInfo': gene,
            'sequenceInfo': seq,
            'type': 'protein'
        }

    return output


def get_ensembl(id: str):

    endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
    resp = requests.get(endpoint, headers={"Content-Type": "application/json"})
    return resp


def ensembl_parse_response(resp):

    data = resp.json()

    if "error" in data:
        return {"error": data["error"]}

    return {
        data.get("id"): {
            "object_type": data.get("object_type"),
            "species": data.get("species"),
            "assembly_name": data.get("assembly_name"),
            "biotype": data.get("biotype"),
            "display_name": data.get("display_name"),
            "id": data.get("id"),
            "db_type": data.get("db_type"),
            "description": data.get("description"),
            "source": data.get("source"),
            "canonical_transcript": data.get("canonical_transcript")
        }
    }


def main(ids):

    results = {}

    uniprot_pattern = r"^[A-Z0-9]{6}$"
    ensembl_pattern = r"^ENS[A-Z]*G\d+"

    for id in ids:

        if re.match(uniprot_pattern, id):

            resp = get_uniprot(id)

            if resp.status_code != 200:
                results[id] = f"error:{resp.status_code}"
                continue

            results.update(uniprot_parse_response(resp))

        elif re.match(ensembl_pattern, id):

            resp = get_ensembl(id)

            if resp.status_code != 200:
                results[id] = f"error:{resp.status_code}"
                continue

            results.update(ensembl_parse_response(resp))

        else:
            results[id] = "error:unknown database"

    return results

In [5]:
get_uniprot('P11473')

<Response [200]>

In [6]:
get_uniprot('helloworld')

<Response [400]>

In [7]:
get_uniprot('helloworld').json()

{'url': 'http://rest.uniprot.org/uniprotkb/accessions',
 'messages': ["Accession 'helloworld' has invalid format. It should be a valid UniProtKB accession with optional sequence range e.g. P12345[10-20]."]}

In [8]:
uniprot_parse_response(get_uniprot('P11473'))

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'}}

In [9]:
get_ensembl('ENSMUSG00000041147')

<Response [200]>

In [4]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'},
 'Q91XI3': {'organism': 'Ictidomys tridecemlineatus',
  'geneInfo': [{'geneName': {'value': 'INS'}}],
  'sequenceInfo': {'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHLVE